In [ ]:
%pip install torch numpy transformers datasets tiktoken wandb tqdm

In [ ]:
# Calculate number of parameters in a model using config from train_rocstories.py
import torch
from model import GPT, GPTConfig
import importlib.util
import pickle

with open('data/rocstories/meta.pkl', 'rb') as f:
    meta = pickle.load(f)
print("Vocab size:", meta['vocab_size'])

# Load config from config/train_rocstories.py
spec = importlib.util.spec_from_file_location("train_rocstories", "config/train_rocstories.py")
config_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(config_mod)

config = GPTConfig(
    n_layer=config_mod.n_layer,
    n_head=config_mod.n_head,
    n_embd=config_mod.n_embd,
    block_size=config_mod.block_size,
    vocab_size=meta['vocab_size'],  # Use vocab_size from meta.pkl
    dropout=config_mod.dropout,
    bias=config_mod.bias
)
model = GPT(config)
num_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters: {num_params:,}")

In [ ]:
import torch
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("No GPU available")

In [ ]:
import tiktoken
import numpy as np

data = np.fromfile('data/rocstories/train.bin', dtype=np.uint16)
enc = tiktoken.get_encoding('gpt2')

text = enc.decode(data[98:100].tolist())
print("Text:", text)
print("Token ID:", enc.encode(text))

In [ ]:
import torch
from pprint import pprint

ckpt = torch.load('out-rocstories/25_83.pt', map_location='cpu')
print("Keys in checkpoint:", ckpt.keys())
for k in ckpt:
    print(f"\nKey: {k}")
    print(f"Type: {type(ckpt[k])}")
    if isinstance(ckpt[k], dict):
        print(f"Subkeys: {list(ckpt[k].keys())}")
    else:
        print(f"Value (truncated): {str(ckpt[k])[:500]}")

In [ ]:
import torch

ckpt = torch.load('out-rocstories-clean/25_18.pt', map_location='cpu')

print("=== Model Args ===")
for k, v in ckpt['model_args'].items():
    print(f"  {k}: {v}")

print("\n=== Config ===")
for k, v in ckpt['config'].items():
    print(f"  {k}: {v}")

print("\n=== Optimizer ===")
print(f"  param_groups: {ckpt['optimizer']['param_groups']}")

print("\n=== Training Info ===")
print(f"  iter_num: {ckpt['iter_num']}")
print(f"  best_val_loss: {ckpt['best_val_loss']}")

=== Model Args ===
  n_layer: 6
  n_head: 6
  n_embd: 384
  block_size: 256
  bias: False
  vocab_size: 50257
  dropout: 0.1

=== Config ===
  out_dir: out-rocstories-clean
  eval_interval: 200
  log_interval: 25
  eval_iters: 100
  eval_only: False
  always_save_checkpoint: True
  init_from: resume
  wandb_log: True
  wandb_project: rocstories-clean
  wandb_run_name: stage3-clean-sft
  dataset: rocstories
  gradient_accumulation_steps: 2
  batch_size: 32
  block_size: 256
  n_layer: 12
  n_head: 12
  n_embd: 768
  dropout: 0.1
  bias: False
  learning_rate: 1e-05
  max_iters: 54000
  weight_decay: 0.1
  beta1: 0.9
  beta2: 0.95
  grad_clip: 1.0
  decay_lr: False
  warmup_iters: 50
  lr_decay_iters: 54000
  min_lr: 1e-05
  backend: nccl
  device: cuda
  dtype: bfloat16
  compile: False

=== Optimizer ===
  param_groups: [{'weight_decay': 0.15, 'lr': 1e-05, 'betas': (0.9, 0.95), 'eps': 1e-08, 'amsgrad': False, 'maximize': False, 'foreach': None, 'capturable': False, 'differentiable': Fa

In [5]:
import torch
import os

checkpoint_dir = 'out-rocstories'
results = []

for fname in os.listdir(checkpoint_dir):
    if fname.endswith('.pt'):
        path = os.path.join(checkpoint_dir, fname)
        try:
            ckpt = torch.load(path, map_location='cpu')
            entry = {
                'file': fname,
                'iter_num': ckpt.get('iter_num', None),
                'best_val_loss': float(ckpt.get('best_val_loss', 0)),
                **ckpt.get('model_args', {})
            }
            results.append(entry)
            print(f"✓ {fname}")
        except Exception as e:
            print(f"✗ {fname}: {e}")

print("\n=== Summary ===")
for r in sorted(results, key=lambda x: x['best_val_loss']):
    print(r)

✓ 25_83.pt
✓ 25_98.pt
✓ 26_21.pt
✓ 26_36.pt
✓ 26_83.pt
✓ 29_94.pt

=== Summary ===
{'file': '26_36.pt', 'iter_num': 4000, 'best_val_loss': 3.2003109455108643, 'n_layer': 24, 'n_head': 16, 'n_embd': 256, 'block_size': 256, 'bias': False, 'vocab_size': 50258, 'dropout': 0.15}
{'file': '26_83.pt', 'iter_num': 8800, 'best_val_loss': 3.243415117263794, 'n_layer': 6, 'n_head': 6, 'n_embd': 384, 'block_size': 128, 'bias': False, 'vocab_size': 50258, 'dropout': 0.2}
{'file': '25_83.pt', 'iter_num': 35600, 'best_val_loss': 3.25311279296875, 'n_layer': 6, 'n_head': 6, 'n_embd': 384, 'block_size': 256, 'bias': False, 'vocab_size': 50257, 'dropout': 0.2}
{'file': '25_98.pt', 'iter_num': 30000, 'best_val_loss': 3.258187770843506, 'n_layer': 6, 'n_head': 6, 'n_embd': 384, 'block_size': 256, 'bias': False, 'vocab_size': 50257, 'dropout': 0.2}
{'file': '26_21.pt', 'iter_num': 18600, 'best_val_loss': 3.2602295875549316, 'n_layer': 6, 'n_head': 6, 'n_embd': 384, 'block_size': 256, 'bias': False, 'vocab_